# Process Binance Spot Monthly Kline ZIP Files

This notebook:
1. Filters monthly ZIP files by `YYYY-MM` range.
2. Extracts CSVs from those ZIP files.
3. Adds spot kline headers (from `README.md`).
4. Merges all extracted monthly CSVs into one combined CSV.


In [ ]:
from pathlib import Path
from datetime import datetime
import csv
import io
import re
from zipfile import ZipFile

# -----------------------------
# Configuration
# -----------------------------
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "downloads").exists() and (PROJECT_ROOT.parent / "downloads").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SYMBOL = "BTCUSDT"
INTERVAL = "1s"
START_YYYY_MM = "2017-01"  # inclusive
END_YYYY_MM = "2020-02"    # inclusive

SOURCE_DIR = PROJECT_ROOT / "downloads" / "data" / "spot" / "monthly" / "klines" / SYMBOL / INTERVAL
OUTPUT_DIR = PROJECT_ROOT / "downloads" / "test1"
OVERWRITE_EXISTING = True

KLINE_COLUMNS = [
    "Open time",
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",
    "Close time",
    "Quote asset volume",
    "Number of trades",
    "Taker buy base asset volume",
    "Taker buy quote asset volume",
    "Ignore",
]

print(f"Project root: {PROJECT_ROOT}")
print(f"Source dir  : {SOURCE_DIR}")
print(f"Output dir  : {OUTPUT_DIR}")


In [ ]:
def parse_yyyy_mm(text: str) -> datetime:
    return datetime.strptime(text, "%Y-%m")


start_dt = parse_yyyy_mm(START_YYYY_MM)
end_dt = parse_yyyy_mm(END_YYYY_MM)
if start_dt > end_dt:
    raise ValueError(f"START_YYYY_MM must be <= END_YYYY_MM ({START_YYYY_MM} > {END_YYYY_MM})")

if not SOURCE_DIR.exists():
    raise FileNotFoundError(f"Source directory not found: {SOURCE_DIR}")

zip_pattern = re.compile(
    rf"^{re.escape(SYMBOL)}-{re.escape(INTERVAL)}-(\d{{4}}-\d{{2}})\.zip$"
)

selected_zips = []
for zip_path in SOURCE_DIR.glob("*.zip"):
    match = zip_pattern.match(zip_path.name)
    if not match:
        continue
    month_text = match.group(1)
    month_dt = parse_yyyy_mm(month_text)
    if start_dt <= month_dt <= end_dt:
        selected_zips.append((month_dt, month_text, zip_path))

selected_zips.sort(key=lambda x: x[0])

if not selected_zips:
    raise ValueError(
        f"No ZIP files matched range {START_YYYY_MM} to {END_YYYY_MM} in {SOURCE_DIR}"
    )

print(f"Selected ZIP files: {len(selected_zips)}")
for i, (_, month_text, zip_path) in enumerate(selected_zips, start=1):
    if i <= 10 or i > len(selected_zips) - 3:
        print(f"  {month_text}: {zip_path.name}")
    elif i == 11:
        print("  ...")


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

extracted_csv_paths = []

for _, month_text, zip_path in selected_zips:
    with ZipFile(zip_path, "r") as zf:
        csv_members = [name for name in zf.namelist() if name.lower().endswith(".csv")]
        if not csv_members:
            raise ValueError(f"No CSV file found in ZIP: {zip_path}")

        if len(csv_members) > 1:
            print(
                f"Warning: {zip_path.name} contains {len(csv_members)} CSV files. "
                "All files will be processed."
            )

        for idx, member in enumerate(csv_members, start=1):
            suffix = "" if len(csv_members) == 1 else f"-part{idx}"
            output_name = f"{SYMBOL}-{INTERVAL}-{month_text}{suffix}.csv"
            output_csv_path = OUTPUT_DIR / output_name

            if output_csv_path.exists() and not OVERWRITE_EXISTING:
                print(f"Skipping existing file (OVERWRITE_EXISTING=False): {output_csv_path}")
                extracted_csv_paths.append(output_csv_path)
                continue

            with zf.open(member, "r") as src_bin, output_csv_path.open(
                "w", newline="", encoding="utf-8"
            ) as dst:
                reader = csv.reader(io.TextIOWrapper(src_bin, encoding="utf-8"))
                writer = csv.writer(dst)

                writer.writerow(KLINE_COLUMNS)

                first_row = next(reader, None)
                if first_row is not None:
                    # Most Binance files do not include a header. If one exists, do not duplicate it.
                    if first_row != KLINE_COLUMNS:
                        writer.writerow(first_row)
                    for row in reader:
                        writer.writerow(row)

            extracted_csv_paths.append(output_csv_path)
            print(f"Wrote: {output_csv_path}")

print(f"Finished extracting {len(extracted_csv_paths)} CSV file(s) to {OUTPUT_DIR}")


In [ ]:
combined_output_path = OUTPUT_DIR / (
    f"{SYMBOL}-{INTERVAL}-{START_YYYY_MM}_to_{END_YYYY_MM}-combined.csv"
)

total_rows = 0
with combined_output_path.open("w", newline="", encoding="utf-8") as dst:
    writer = csv.writer(dst)
    writer.writerow(KLINE_COLUMNS)

    for csv_path in extracted_csv_paths:
        with csv_path.open("r", newline="", encoding="utf-8") as src:
            reader = csv.reader(src)
            first_row = next(reader, None)
            if first_row is None:
                continue

            if first_row != KLINE_COLUMNS:
                writer.writerow(first_row)
                total_rows += 1

            for row in reader:
                writer.writerow(row)
                total_rows += 1

print(f"Combined CSV: {combined_output_path}")
print(f"Monthly CSV files merged: {len(extracted_csv_paths)}")
print(f"Total data rows in combined CSV: {total_rows}")


In [ ]:
print("Preview (first 6 lines):")
with combined_output_path.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        print(line.rstrip())
        if i >= 5:
            break
